# 3. Area Of Interest

Tutorial 2 narrowed the data by *value* and by *label*. This one narrows it by
**geography**: a box, a circle, a polygon, or a vertical cross-section.

| method | AOI | typical use |
|---|---|---|
| `crop_by_bbox` | rectangle | a map tile, a model domain |
| `crop_around_point` | circle | everything within N km of a place |
| `crop_by_polygone` | any polygon | a catchment, a shapefile |
| `extract_cross_section` | a line + beam width | a vertical slice through a storm |

The first three keep the gates and drop the rest. The fourth also computes, for
every selected gate, its position **along the line** and its **altitude** (the
geometry a vertical cross-section plot needs).

---
## One rule about the crs

Every AOI runs in the **archive's own CRS**, read from `info.yaml`. You never have
to restate it. What you *do* have to say is which CRS **your own** coordinates are
in, using the argument `crs=`:

```python
rdf.crop_around_point(point=(8.83, 46.04), distance=30_000, crs=4326)   # lon/lat
rdf.crop_around_point(point=(2680000, 1120000), distance=30_000)        # already LV95
```

Get this wrong and the crop is silently empty or in the wrong country (passing
lon/lat degrees while RadDB reads them as metres puts your AOI ~2600 km away).

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import shapely

import raddb

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these three paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.  If it has not run,
# the cell below builds it.

MCH_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/MCH_datatree").expanduser()
NEXRAD_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

In [ ]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056)
rdf = db.open(radars="L").filter({"var": "DBZH", "logic": ">", "threshold": 5})

info = db.get_radar_info("L")
SITE = (info["longitude"], info["latitude"])  # radar: L ==> lon, lat
print(f"radar L at {SITE[0]:.3f}, {SITE[1]:.3f}   |   {len(rdf):,} gates with echo")
print("archive CRS:", rdf.crs())

### `crs=` describes *your* numbers

This is the single most common mistake, so it is worth seeing rather than reading.
`crs=` says which frame **the coordinates you passed in** are expressed in. It does
**not** choose the frame the AOI runs in — that is always the archive's own CRS.

So `crs=4326` throughout this notebook because the points are written as lon/lat
degrees. Passing `crs=2056` with those same numbers tells RadDB to read `8.83` and
`46.04` as *metres* in LV95 — a spot near the origin of the Swiss grid, ~2700 km
from the radar — and the crop comes back empty.

In [ ]:
from pyproj import Transformer

# The radar site, written both ways
E, N = Transformer.from_crs(4326, 2056, always_xy=True).transform(*SITE)
print(f"lon/lat (EPSG:4326): ({SITE[0]:.4f}, {SITE[1]:.4f})")
print(f"LV95    (EPSG:2056): ({E:,.0f}, {N:,.0f})\n")

print("crop_around_point(distance=30 km):")
print(f"  (lon, lat) + crs=4326 -> {len(rdf.crop_around_point(point=SITE, distance=30_000, crs=4326)):>8,} gates")
print(
    f"  (lon, lat) + crs=2056 -> {len(rdf.crop_around_point(point=SITE, distance=30_000, crs=2056)):>8,} gates"
    "  <- degrees read as metres",
)
print(f"  (E, N)     + crs=2056 -> {len(rdf.crop_around_point(point=(E, N), distance=30_000, crs=2056)):>8,} gates")
print(
    f"  (E, N)     + no crs   -> {len(rdf.crop_around_point(point=(E, N), distance=30_000)):>8,} gates"
    "  <- defaults to the archive CRS",
)

## 1. `crop_by_bbox` — a rectangle

Pass either `bounds=(minx, miny, maxx, maxy)` or `extent=(minx, maxx, miny, maxy)`
— the latter matches matplotlib's `ax.axis()` ordering.

In [ ]:
box = rdf.crop_by_bbox(bounds=(8.4, 45.8, 9.3, 46.5), crs=4326)
print(f"{len(rdf):,} -> {len(box):,} gates inside the lon/lat box")
print("lon/lat extent of the result:", [round(v, 3) for v in box.geographic_extent()])

## 2. `crop_around_point` — everything within N km

`distance` is in **metres**, measured in the archive's projection — so it is a true
ground distance, not a coordinate difference.

In [ ]:
for km in (20, 50, 100):
    sub = rdf.crop_around_point(point=SITE, distance=km * 1_000, crs=4326)
    print(f"  within {km:>3} km : {len(sub):>8,} gates")

In [ ]:
# Any point, not only the radar itself
elsewhere = rdf.crop_around_point(point=(9.0, 46.2), distance=25_000, crs=4326)
print(f"25 km around (9.0, 46.2): {len(elsewhere):,} gates")

## 3. `crop_by_polygone` — an arbitrary shape

Accepts a shapely `Polygon`/`MultiPolygon`, a GeoDataFrame, or a path to a
`.shp` / `.geojson` file. A file's **declared CRS wins** unless you pass `crs=`
explicitly.

In [ ]:
triangle = shapely.Polygon([(8.6, 45.9), (9.2, 46.1), (8.8, 46.5)])
poly = rdf.crop_by_polygone(polygon=triangle, crs=4326)
print(f"inside the triangle: {len(poly):,} gates")

In [ ]:
# The same thing from a file on disk
import json

geojson_path = ARCHIVE_DIR / "aoi_demo.geojson"
geojson_path.write_text(
    json.dumps(
        {
            "type": "FeatureCollection",
            "features": [{"type": "Feature", "properties": {}, "geometry": shapely.geometry.mapping(triangle)}],
        },
    ),
)

from_file = rdf.crop_by_polygone(polygon=geojson_path)  # CRS taken from the file
print(f"from GeoJSON: {len(from_file):,} gates  (same: {len(from_file) == len(poly)})")

## 4. `extract_cross_section`: a vertical cross-section

A line `p1 -> p2` plus the beam width defines a vertical curtain. Every gate whose
beam intersects it is kept, and gains the geometry of the section:

| column | meaning |
|---|---|
| `d_center` | the gate centre's distance **along the line** from `p1` [m] |
| `z_center` | the gate centre's altitude above sea level [m] |
| `cs_polygon` | the gate's footprint in the (distance, altitude) plane |

Unlike an area crop, these columns are **not** LUT data — they belong to this
particular line, so they travel with the rows.

`cs_polygon` is the one `plot_vcs` draws; `d_center` / `z_center` are there for
analysing a section numerically — profiles, height thresholds, distance bins.

In [ ]:
cs = rdf.extract_cross_section(
    p1=(SITE[0] - 0.6, SITE[1] - 0.35),
    p2=(SITE[0] + 0.6, SITE[1] + 0.35),
    crs=4326,
)
print(f"{len(cs):,} gates on the section")
# in new columns it should appear: 'd_center', 'z_center', 'cs_polygon'
print("new columns:", [c for c in cs.columns() if c not in rdf.columns()])

### Reading `cs_polygon`

Inside the polars frame `cs_polygon` is a `Binary` column — the polygon
**WKB-encoded**, because polars has no geometry dtype. That is why printing
`.data` shows bytes rather than coordinates.

It is decoded back to a real shapely `Polygon` the moment you leave polars:

In [ ]:
print("in polars   :", cs.data.schema["cs_polygon"])
print("raw value   :", str(cs.data["cs_polygon"][0])[:40], "...")

# to_pandas() decodes it; to_geopandas() does too
poly = cs.to_pandas()["cs_polygon"].iloc[0]
print("\nafter to_pandas:", type(poly).__name__)
print("  WKT     :", poly.wkt[:90], "...")
print("  corners :", list(poly.exterior.coords)[:2], "...")
print("  bounds  :", tuple(round(v) for v in poly.bounds), "(d_min, z_min, d_max, z_max)")

## 5. `quicklook=True` — did I crop what I meant to?

Every AOI method takes `quicklook=True`, which draws the AOI footprint and the
selected gates on a country-scale map. It is a sanity check, not a product.

In [ ]:
_ = rdf.crop_around_point(point=SITE, distance=50_000, crs=4326, quicklook=True)
plt.show()

In [ ]:
_ = rdf.crop_by_polygone(polygon=triangle, crs=4326, quicklook=True)
plt.show()

## 6. Chaining AOIs

Crops return a RadDB like everything else, so they compose with `filter` and `sel`
and with each other.

In [ ]:
storm = (
    db.open(radars="L")
    .filter({"var": "DBZH", "logic": ">", "threshold": 35})
    .crop_around_point(point=SITE, distance=60_000, crs=4326)
    .sel(range=slice(5_000, 60_000))
)
print(f"{len(storm):,} gates: strong echo, within 60 km, beyond 5 km range")

## 7. The interactive tool

In Jupyter, `interactive_crop()` puts an [ipyleaflet](https://ipyleaflet.readthedocs.io/)
map in front of you: draw a shape with the toolbar, click **Apply crop**, and the
crop runs for you.

The tool dispatches on what you drew:

| you draw | RadDB runs |
|---|---|
| rectangle | `crop_by_bbox` |
| polygon | `crop_by_polygone` |
| marker | `crop_around_point`, using the **radius** box under the map |
| **polyline** | `extract_cross_section` |

It returns a **selector object**, and the crop lands on its attributes:

| attribute | what it holds |
|---|---|
| `selector.result` | the cropped **RadDB** — the same object any `crop_*` returns, so it filters, plots and converts like the rest of this notebook |
| `selector.kind` | which crop ran: `'bbox'`, `'polygon'`, `'point'` or `'cross_section'` |
| `selector.feature` | the shape you drew, as GeoJSON in lon/lat |

`selector.result` stays `None` until you press *Apply crop*, which is why drawing
and reading the result are split across the next two cells.

In [ ]:
RUN_INTERACTIVE = True  # <- set False to skip the map (e.g. outside Jupyter)

if RUN_INTERACTIVE:
    # `selector` is the widget handle — the map appears immediately below.
    # Draw a shape with the toolbar, then click "Apply crop" before running the
    # next cell.  For a marker, set the radius box first; it is read on click.
    selector = rdf.interactive_crop()
else:
    selector = None
    print("interactive_crop() needs a live Jupyter kernel ==> set RUN_INTERACTIVE = True")

In [ ]:
from IPython.display import display

# Run this cell *after* drawing a shape above and clicking "Apply crop".
# `cropped` is an ordinary RadDB: filter it, crop it again, plot it or convert it,
# exactly as in the sections above.  Until Apply crop is pressed it is still None.
if RUN_INTERACTIVE and selector is not None and selector.result is not None:
    cropped = selector.result  # <- the cropped RadDB
    print(f"{selector.kind} -> {len(cropped):,} gates")
    print("drawn shape :", selector.feature["geometry"]["type"])
    display(cropped.head())
else:
    print("nothing applied yet — draw a shape above, click 'Apply crop', then re-run this cell")

---
**Next:** [4 — Plots](04_plots.ipynb)